In [ ]:
# =========================
# INSTALL LIBRARIES
# =========================
!pip install -q datasets transformers evaluate scikit-learn torch

# =========================
# STEP 1: LOAD DATASET
# =========================
from datasets import load_dataset

dataset = load_dataset("sms_spam")
print("Columns:", dataset["train"].column_names)
print("Labels:", dataset["train"].features["label"].names)

# =========================
# STEP 2: LABEL MAPS
# =========================
label_map = {0: "ham", 1: "spam"}
id_map = {"ham": 0, "spam": 1}

# =========================
# STEP 3: TOKENIZATION
# =========================
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fn(examples):
    return tokenizer(
        examples["sms"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized = dataset.map(tokenize_fn, batched=True)

# =========================
# STEP 4: SHUFFLE & SPLIT
# =========================
data = tokenized["train"].shuffle(seed=42)

train_data = data.select(range(5000))
eval_data = data.select(range(5000, len(data)))

train_data = train_data.remove_columns(["sms"])
eval_data = eval_data.remove_columns(["sms"])

train_data.set_format("torch")
eval_data.set_format("torch")

# =========================
# STEP 5: TRAIN MODEL
# =========================
import numpy as np
import evaluate
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    compute_metrics=compute_metrics,  # ✅ tokenizer REMOVED
)

trainer.train()

# =========================
# STEP 6: SAVE MODEL
# =========================
trainer.save_model("./spam_model")
tokenizer.save_pretrained("./spam_model")

# =========================
# STEP 7: LOAD & PREDICT
# =========================
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("./spam_model")
tokenizer = AutoTokenizer.from_pretrained("./spam_model")
model.eval()

def predict_with_label(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )
    with torch.no_grad():
        outputs = model(**inputs)

    pred_id = torch.argmax(outputs.logits, dim=1).item()
    return label_map[pred_id]

# =========================
# TEST
# =========================
texts = [
    "Congratulations! You've won a free ticket.",
    "Hey, are we meeting tomorrow?",
]

for t in texts:
    print(t, "→", predict_with_label(t))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Congratulations! You've won a free ticket. → ham
Hey, are we meeting tomorrow? → ham


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load trained model
model = AutoModelForSequenceClassification.from_pretrained("./spam_model")
tokenizer = AutoTokenizer.from_pretrained("./spam_model")
model.eval()

label_map = {0: "ham", 1: "spam"}

def predict_with_label(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )
    with torch.no_grad():
        outputs = model(**inputs)

    pred_id = torch.argmax(outputs.logits, dim=1).item()
    return label_map[pred_id]

# =========================
# TEST CASES (ONE GO)
# =========================
test_messages = [
    "Congratulations! You won 50,000 PKR cash prize",
    "Hey, are we meeting tomorrow?",
    "URGENT! Click the link to claim your reward",
    "Let's go for dinner tonight",
    "Free entry in a weekly lucky draw"
]

for msg in test_messages:
    print(f"Message: {msg}")
    print("Prediction:", predict_with_label(msg))
    print("-" * 50)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Message: Congratulations! You won 50,000 PKR cash prize
Prediction: spam
--------------------------------------------------
Message: Hey, are we meeting tomorrow?
Prediction: ham
--------------------------------------------------
Message: URGENT! Click the link to claim your reward
Prediction: spam
--------------------------------------------------
Message: Let's go for dinner tonight
Prediction: ham
--------------------------------------------------
Message: Free entry in a weekly lucky draw
Prediction: spam
--------------------------------------------------
